In [1]:
import pymupdf4llm
import pdfplumber

In [2]:
knowledge_base1 = "/Users/furkanmelih/personal_projects/bgts-case/static/knowledge_base/KB-01_BGP_Troubleshooting.pdf"
pdf = pdfplumber.open(knowledge_base1)

In [3]:
pg1 = pdf.pages[0]

In [4]:
print(pg1.extract_text())

KB-01: BGP Troubleshooting ve RCA Kılavuzu
Versiyon: 2.3 | Son Güncelleme: 2024-01-15 | Ekip: Network Operasyon
1. BGP Durum Makinesi
BGP oturumu aşağıdaki durumlar üzerinden geçer. Her geçiş bir event tetikler:
Durum Açıklama Beklenen Süre Sorun Sinyali
Idle Başlangıç durumu, bağlantı bekleniyor < 5 sn Sürekli Idle kalıyorsa ACL/route problemi
Connect TCP bağlantısı kuruluyor < 10 sn Timeout → firewall bloğu
Active TCP başarısız, yeniden deneniyor < 30 sn Loop → neighbor IP yanlış
OpenSent OPEN mesajı gönderildi < 5 sn Uzun kalıyorsa AS numarası uyuşmuyor
OpenConfirm KEEPALIVE bekleniyor < 5 sn Hold timer mismatch
Established Oturum aktif, prefix alışverişi Sürekli Düşerse → log analizi şart
2. Yaygın BGP Hata Logları ve Anlamları
%BGP-5-ADJCHANGE: neighbor 195.142.11.1 Down BGP Notification sent
--> Notification sent: bizim tarafimiz oturumu kapatti. Neden: hold timer expired.
%BGP-3-NOTIFICATION: sent to neighbor 10.0.0.1 4/0 (hold time expired) 0 bytes
--> Keepalive paketleri karsi

In [5]:
pg1.extract_tables()[0]

[['Durum', 'Açıklama', 'Beklenen Süre', 'Sorun Sinyali'],
 ['Idle',
  'Başlangıç durumu, bağlantı bekleniyor',
  '< 5 sn',
  'Sürekli Idle kalıyorsa ACL/route problemi'],
 ['Connect',
  'TCP bağlantısı kuruluyor',
  '< 10 sn',
  'Timeout → firewall bloğu'],
 ['Active',
  'TCP başarısız, yeniden deneniyor',
  '< 30 sn',
  'Loop → neighbor IP yanlış'],
 ['OpenSent',
  'OPEN mesajı gönderildi',
  '< 5 sn',
  'Uzun kalıyorsa AS numarası uyuşmuyor'],
 ['OpenConfirm', 'KEEPALIVE bekleniyor', '< 5 sn', 'Hold timer mismatch'],
 ['Established',
  'Oturum aktif, prefix alışverişi',
  'Sürekli',
  'Düşerse → log analizi şart']]

In [6]:
pdf.close()

In [7]:
markdown = pymupdf4llm.to_markdown(knowledge_base1)

=== Document parser messages ===
Using Tesseract for OCR processing.



In [8]:
print(markdown)

## **KB-01: BGP Troubleshooting ve RCA Kılavuzu** 

_Versiyon: 2.3 | Son Güncelleme: 2024-01-15 | Ekip: Network Operasyon_ 

## **1. BGP Durum Makinesi** 

BGP oturumu aşağıdaki durumlar üzerinden geçer. Her geçiş bir event tetikler: 

|**Durum**|**Açıklama**|**Beklenen Süre**|**Sorun Sinyali**|
|---|---|---|---|
|Idle|Başlangıç durumu, bağlantı bekleniyor|< 5 sn|Sürekli Idle kalıyorsa ACL/route problemi|
|Connect|TCP bağlantısı kuruluyor|< 10 sn|Timeout → firewall bloğu|
|Active|TCP başarısız, yeniden deneniyor|< 30 sn|Loop → neighbor IP yanlış|
|OpenSent|OPEN mesajı gönderildi|< 5 sn|Uzun kalıyorsa AS numarası uyuşmuyor|
|OpenConfirm|KEEPALIVE bekleniyor|< 5 sn|Hold timer mismatch|
|Established|Oturum aktif, prefix alışverişi|Sürekli|Düşerse → log analizi şart|



## **2. Yaygın BGP Hata Logları ve Anlamları** 

```
%BGP-5-ADJCHANGE: neighbor 195.142.11.1 Down BGP Notification sent
--> Notification sent: bizim tarafimiz oturumu kapatti. Neden: hold timer expired.
```

```
%BGP-3-NOTI

In [9]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader

In [10]:
with open("markdown.md", "w") as f:
    f.write(markdown)

loader = UnstructuredMarkdownLoader(file_path="markdown.md", mode="elements")
docs = loader.load()

In [ ]:
table_docs = []
text_docs = []

current_doc_title = "Unknown Document"

for doc in docs:
    # Unstructured tags each chunk with a category (e.g., 'Table', 'Title', 'NarrativeText')
    category = doc.metadata.get("category")

     # Whenever we hit a header, update our tracking variables
    if category == "Title":
        # Unstructured often includes 'category_depth' to distinguish H1, H2, H3
        # If your version doesn't, we default to treating it as a section.
        depth = doc.metadata.get("category_depth", 1)
        if depth == 1:       # Represents an H1 (#)
            current_doc_title = doc.page_content
        else:
            raise ValueError(f"Unexpected header depth: {depth}")

    if category == "Table":
        doc.metadata["doc_title"] = current_doc_title
        table_docs.append(doc)
        # You can access the HTML version of the table like this:
        html_table = doc.metadata.get("text_as_html")
        print("--- Found a Table ---")
        print(f"HTML format: {html_table[:100]}...") # Printing a snippet
    else:
        text_docs.append(doc)



--- Found a Table ---
HTML format: <table><tr><td>Durum</td><td>Açıklama</td><td>Beklenen Süre</td><td>Sorun Sinyali</td></tr><tr><td>I...
--- Found a Table ---
HTML format: <table><tr><td>Komut</td><td>Amaç</td><td>Kritik Çıktı</td></tr><tr><td>show bgp summary</td><td>Tüm...


In [28]:
### YENİ YAKLAŞIM

from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)
from langchain_core.documents import Document
from loguru import logger


CHUNK_SIZE = 1024
TEXT_CHUNK_OVERLAP = 100

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ],
    strip_headers=True,
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=TEXT_CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)


def derive_doc_title(meta: dict) -> str:
    """Pick the most specific heading available as the doc_title."""
    for key in ("h3", "h2", "h1"):
        if meta.get(key):
            return meta[key]
    return "Unknown Document"


def load_split_documents(file_path: str) -> tuple[list[Document], list[Document]]:
    """
    - Tables: UnstructuredMarkdownLoader(mode='elements') — keeps text_as_html
    - Narrative: raw markdown -> MarkdownHeaderTextSplitter (semantic) ->
                 RecursiveCharacterTextSplitter (size) happens later in chunk_documents
    """
    # 1) Tables from elements mode
    el_loader = UnstructuredMarkdownLoader(file_path=file_path, mode="elements")
    el_docs = el_loader.load()

    table_docs: list[Document] = []
    current_section_title = "Unknown Document"

    for doc in el_docs:
        category = doc.metadata.get("category")

        if category == "Title":
            current_section_title = doc.page_content.strip().strip("*").strip()
            continue

        if category == "Table":
            doc.metadata["doc_title"] = current_section_title
            table_docs.append(doc)

    # 2) Narrative from raw markdown via header splitter
    with open(file_path, "r", encoding="utf-8") as f:
        raw_md = f.read()

    header_chunks = header_splitter.split_text(raw_md)

    text_docs: list[Document] = []
    for hc in header_chunks:
        body = hc.page_content.strip()
        if not body:
            continue

        # Tables already covered by elements loader — strip them from narrative
        import re
        body = re.sub(r"^\|.*\n\|[\s\-:|]+\|\n(?:\|.*\n?)+", "\n", body, flags=re.MULTILINE)
        body = re.sub(r"\n{3,}", "\n\n", body).strip()
        if not body:
            continue

        text_docs.append(Document(
            page_content=body,
            metadata={
                **hc.metadata,  # h1, h2, h3 already populated
                "doc_title": derive_doc_title(hc.metadata),
                "source": file_path,
            },
        ))

    logger.info(
        "Loaded {} table docs, {} narrative sections from {}",
        len(table_docs), len(text_docs), file_path,
    )
    return table_docs, text_docs

In [30]:
table_docs, text_docs = load_split_documents(file_path="markdown.md")

2026-05-10 18:13:17.009 | INFO     | __main__:load_split_documents:92 - Loaded 2 table docs, 5 narrative sections from markdown.md


In [33]:
len(text_docs)

5

In [54]:
import json
print(json.dumps(text_docs[4].metadata, indent=2, ensure_ascii=False))

{
  "h2": "**5. Önleyici Tedbirler**",
  "doc_title": "**5. Önleyici Tedbirler**",
  "source": "markdown.md"
}


In [55]:
print(text_docs[4].page_content)

- BFD (Bidirectional Forwarding Detection) etkinleştir: hold timer 3 sn altına iner  
- BGP community ile route tagging yap, failover politikasını netleştir  
- maximum-prefix limiti ISP ile mutabık kalınan değerin %80'ine ayarla  
- Değişiklik öncesi 'show bgp summary' çıktısını kaydet (baseline)


In [56]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

CHUNK_SIZE = 1024
TEXT_CHUNK_OVERLAP = 100
TABLE_CHUNK_OVERLAP = 0

# --- For narrative text: paragraph -> sentence -> word, with overlap ---
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=TEXT_CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)


In [57]:
from bs4 import BeautifulSoup
from loguru import logger 
from io import StringIO
import pandas as pd


def is_simple_table(html_table: str) -> bool:
    """
    Returns True if the table is processable (no nested tables inside),
    False if it contains another table within it.
    """
    soup = BeautifulSoup(html_table, "html.parser")
    nested_count = len(soup.find_all("table"))

    if nested_count > 1:
        logger.info("Nested table detected (found %d <table> elements).", nested_count)
        return False

    logger.debug("Simple table detected.")
    return True


def table_to_kv_lines(html_table: str) -> tuple[list[str], list[str]]:
    """
    Convert a simple 2D table into 'col1: v1, col2: v2, ...' lines.
    Returns the row lines and the column names (used as header context).
    """
    df = pd.read_html(StringIO(html_table))[0]

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " / ".join(str(p) for p in col if str(p) != "nan").strip()
            for col in df.columns
        ]
    else:
        df.columns = [str(c) for c in df.columns]


    df = df.fillna("")
    cols = [str(c) for c in df.columns]

    lines = [", ".join(f"{c}: {row[c]}" for c in cols) for _, row in df.iterrows()]
    return lines, cols


def split_kv_table(
    lines: list[str],
    cols: list[str],
    doc_title: str,
    extra_meta: dict,
) -> list[Document]:
    """
    Split KV-formatted table rows using RecursiveCharacterTextSplitter
    (prefers '\n' so rows stay intact). Each resulting chunk is prefixed
    with the header context (doc_title + columns) so it remains meaningful
    when retrieved on its own.
    """
    header_ctx = (
        f"[Document: {doc_title}]\n"
        f"[Columns: {', '.join(cols)}]\n"
    )

    body = "\n".join(lines)

    effective_size = max(CHUNK_SIZE - len(header_ctx), 256)

    table_splitter = RecursiveCharacterTextSplitter(
        chunk_size=effective_size,
        chunk_overlap=TABLE_CHUNK_OVERLAP,
        separators=["\n", ", ", " ", ""],
        length_function=len,
    )

    raw_chunks = table_splitter.split_text(body)

    return [
        Document(
            page_content=header_ctx + ch,
            metadata={
                "doc_title": doc_title,
                "type": "table_chunk",
                **extra_meta,
            },
        )
        for ch in raw_chunks
    ]



In [58]:
def chunk_documents(
    table_docs: list[Document],
    text_docs: list[Document],
) -> list[Document]:
    """
    Entrypoint: turn raw Unstructured table/text Documents into indexable chunks.

    - Simple 2D tables -> KV lines, split with no overlap, header context prepended.
    - Tables containing a nested table -> kept as-is (not split).
    - Narrative text -> RecursiveCharacterTextSplitter with overlap.

    Each output chunk has metadata['type'] in {'table_chunk', 'text_chunk'} and
    metadata['doc_title'].
    """
    chunks: list[Document] = []

    # --- Texts ---
    for tdoc in text_docs:
        if not tdoc.page_content.strip():
            continue

        doc_title = tdoc.metadata.get("doc_title", "Unknown Document")
        pieces = text_splitter.split_text(tdoc.page_content)

        for p in pieces:
            chunks.append(
                Document(
                    page_content=p,
                    metadata={
                        **tdoc.metadata,
                        "doc_title": doc_title,
                        "type": "text_chunk",
                    },
                )
            )

    # --- Tables ---
    for tdoc in table_docs:
        html_table = tdoc.metadata.get("text_as_html") or ""
        doc_title = tdoc.metadata.get("doc_title", "Unknown Document")

        if not is_simple_table(html_table):
            chunks.append(tdoc)
            continue

        lines, cols = table_to_kv_lines(html_table)
        if not lines:
            logger.warning("Table produced no rows; skipping. doc_title={}", doc_title)
            continue

        table_chunks = split_kv_table(
            lines=lines,
            cols=cols,
            doc_title=doc_title,
            extra_meta={"source_category": "Table"},
        )
        chunks.extend(table_chunks)

    

    logger.info(
        "Produced {} chunks (from {} table docs, {} text docs).",
        len(chunks),
        len(table_docs),
        len(text_docs),
    )
    return chunks

In [59]:
chks = chunk_documents(table_docs=table_docs, text_docs=text_docs)

2026-05-10 18:19:03.386 | DEBUG    | __main__:is_simple_table:19 - Simple table detected.
2026-05-10 18:19:03.402 | DEBUG    | __main__:is_simple_table:19 - Simple table detected.
2026-05-10 18:19:03.407 | INFO     | __main__:chunk_documents:61 - Produced 7 chunks (from 2 table docs, 5 text docs).


In [72]:

import json
print(json.dumps(chks[6].model_dump(), indent=2, ensure_ascii=False))

{
  "id": null,
  "metadata": {
    "doc_title": "3. Tanılama Komutları (Cisco IOS/IOS-XE)",
    "type": "table_chunk",
    "source_category": "Table"
  },
  "page_content": "[Document: 3. Tanılama Komutları (Cisco IOS/IOS-XE)]\n[Columns: 0, 1, 2]\n0: Komut, 1: Amaç, 2: Kritik Çıktı\n0: show bgp summary, 1: Tüm neighbor durumu, 2: State sütunu: Established?\n0: show bgp neighbors 195.x.x.x, 1: Detaylı neighbor bilgisi, 2: BGP state, hold time, prefix count\n0: show bgp neighbors X advertised-routes, 1: Gönderilen prefixler, 2: Route policy doğru çalışıyor mu?\n0: show bgp neighbors X received-routes, 1: Alınan prefixler, 2: route-map in uygulandı mı?\n0: debug ip bgp 195.x.x.x events, 1: Canlı event izleme, 2: SADECE üretimde dikkatli kullan\n0: clear ip bgp 195.x.x.x soft, 1: Soft reset (trafiği kesmez), 2: Policy değişikliği sonrası",
  "type": "Document"
}


In [69]:
print(chks[5].model_dump()["page_content"])

[Document: 1. BGP Durum Makinesi]
[Columns: 0, 1, 2, 3]
0: Durum, 1: Açıklama, 2: Beklenen Süre, 3: Sorun Sinyali
0: Idle, 1: Başlangıç durumu, bağlantı bekleniyor, 2: < 5 sn, 3: Sürekli Idle kalıyorsa ACL/route problemi
0: Connect, 1: TCP bağlantısı kuruluyor, 2: < 10 sn, 3: Timeout → firewall bloğu
0: Active, 1: TCP başarısız, yeniden deneniyor, 2: < 30 sn, 3: Loop → neighbor IP yanlış
0: OpenSent, 1: OPEN mesajı gönderildi, 2: < 5 sn, 3: Uzun kalıyorsa AS numarası uyuşmuyor
0: OpenConfirm, 1: KEEPALIVE bekleniyor, 2: < 5 sn, 3: Hold timer mismatch
0: Established, 1: Oturum aktif, prefix alışverişi, 2: Sürekli, 3: Düşerse → log analizi şart
